# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [96]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, mean_absolute_percentage_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import optuna

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [ ]:
# Importing the cleaned dataset (No feature enegineering yet! We'll get to that later.)
df_cleaned = pd.read_csv("housing_nofe_update.csv")
q97 = df_cleaned["taxvaluedollarcnt"].quantile(0.97)

# Apply the filter to the full dataset (Cutting the top 3% of outliers)
df_cleaned = df_cleaned[df_cleaned["taxvaluedollarcnt"] <= q97]

In [ ]:
X = df_cleaned.drop("taxvaluedollarcnt", axis=1)
y = df_cleaned['taxvaluedollarcnt']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=random_state)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [12]:
print(X_train_scaled.shape)
print(y_train.shape)

(55722, 32)
(55722,)


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


### Model 1: Linear Regression

In [ ]:
#Setting a CV constant. RepeatedKFold with 5 repeats and 5 splits. 
cv = RepeatedKFold(n_repeats=5, n_splits=5, random_state=random_state)

In [ ]:
# Add as many cells as you need

#Sara's model: Linear Regression
linear_model = LinearRegression()

cv_linear_scores = cross_val_score(linear_model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')

print("Linear Regression Baseline: Scores")
print(f"Mean: {-cv_linear_scores.mean():.4f}")
print(f"Std: {cv_linear_scores.std():.4f}")


Linear Regression Baseline: Scores
Mean: 169628.5406
Std: 1925.2062


### Model 2: Random Forest Regressor

In [34]:
rf_model = RandomForestRegressor(random_state=random_state, n_jobs=-1)

cv_rf_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')

print("Random Forest Regression Baseline: Scores")
print(f"Mean: {-cv_rf_scores.mean():.4f}")
print(f"Std: {cv_rf_scores.std():.4f}")

Random Forest Regression Baseline: Scores
Mean: 148038.5385
Std: 1334.6883


### Model 3: Gradient Boosting (Using XGBoost)

In [35]:
boost_model = xgb.XGBRegressor(random_state=random_state)

cv_boost_scores = cross_val_score(boost_model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')

print("Gradient Boosting/XGBoost Regression Baseline: Scores")
print(f"Mean: {-cv_boost_scores.mean():.4f}")
print(f"Std: {cv_boost_scores.std():.4f}")

Gradient Boosting/XGBoost Regression Baseline: Scores
Mean: 146799.0792
Std: 1147.6865


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Your text here

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


### Adding all agreed features from Milestone 1, plus a few more:

In [36]:
df_fe = df_cleaned.copy()

df_fe['propage'] = df_fe['assessmentyear'] - df_fe['yearbuilt']

df_fe['livingarearatio'] = df_fe['calculatedfinishedsquarefeet'] / df_fe['lotsizesquarefeet']

df_fe['calculatedfinishedsquarefeet_log'] = np.log1p(df_fe['calculatedfinishedsquarefeet'])

df_fe['roomdensity'] = df_fe['roomcnt'] / (df_fe['calculatedfinishedsquarefeet'] + 1)

df_fe['bed_bath_interaction'] = df_fe['bedroomcnt'] * df_fe['bathroomcnt']

df_fe['bath_per_bed'] = df_fe['bathroomcnt'] / (df_fe['bedroomcnt'] + 1)

df_fe['bedroom_intensity'] = df_fe['bedroomcnt'] / (df_fe['calculatedfinishedsquarefeet'] + 1)

In [37]:
df_fe[['propage', 'livingarearatio', 'calculatedfinishedsquarefeet_log', 'roomdensity', 'bed_bath_interaction'
          , 'bath_per_bed','bedroom_intensity']].sample(8)

,propage,livingarearatio,calculatedfinishedsquarefeet_log,roomdensity,bed_bath_interaction,bath_per_bed,bedroom_intensity
47507,62.0,0.183984,7.014814,0.000000,3.0,0.250000,0.002695
6531,48.0,0.041040,7.632886,0.000000,6.0,1.000000,0.000969
45270,43.0,0.406015,7.208600,0.004441,6.0,0.500000,0.002221
23857,33.0,0.154683,7.024649,0.000000,6.0,0.500000,0.002669
67645,42.0,0.021979,7.117206,0.000000,4.0,0.666667,0.001622
15958,48.0,0.220803,7.372118,0.004400,8.0,0.400000,0.002514
45961,14.0,0.133969,7.479300,0.000000,4.0,0.666667,0.001129
27774,71.0,0.199164,6.908755,0.000000,6.0,1.000000,0.001998


#### Due to variables like `propage` interacting with values like `yearbuilt` and `assessmentyear` (possible multicollineraity), we'll be removing these out of the picture. 

In [38]:
df_fe = df_fe.drop(columns=['yearbuilt', 'assessmentyear'], axis=1)

#### Scaling the Feature Engineered values

In [39]:
X_fe = df_fe.drop("taxvaluedollarcnt", axis=1)
y_fe = df_fe['taxvaluedollarcnt']

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y_fe, train_size=0.8, random_state=random_state)

scaler = StandardScaler()
X_train_fe_scaled = scaler.fit_transform(X_train_fe)
X_test_fe_scaled = scaler.fit_transform(X_test_fe)

### Model 1

In [43]:
cv_linear_fe_scores = cross_val_score(linear_model, X_train_fe_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Linear Regression Baseline with Feature Engineering: Scores")
print(f"Mean MAE: {-cv_linear_fe_scores.mean():.4f}")
print(f"Std Deviation: {cv_linear_fe_scores.std():.4f}")

Linear Regression Baseline with Feature Engineering: Scores
Mean MAE: 169274.5684
Std Deviation: 1952.8076


### Model 2

In [44]:
cv_rf_fe_scores = cross_val_score(rf_model, X_train_fe_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Random Forest Regression Baseline with Feature Engineering: Scores")
print(f"Mean: {-cv_rf_fe_scores.mean():.4f}")
print(f"Std: {cv_rf_fe_scores.std():.4f}")

Random Forest Regression Baseline with Feature Engineering: Scores
Mean: 148123.7961
Std: 1292.5375


### Model 3

In [45]:
cv_boost_fe_scores = cross_val_score(boost_model, X_train_fe_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Gradient Boosting/XGBoost Regression Baseline with Feature Engineering: Scores")
print(f"Mean: {-cv_boost_fe_scores.mean():.4f}")
print(f"Std: {cv_boost_fe_scores.std():.4f}")

Gradient Boosting/XGBoost Regression Baseline with Feature Engineering: Scores
Mean: 147215.1120
Std: 1062.3372


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




> Your text here

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [77]:
X_train_fe_linear = X_train_fe_scaled.copy()
X_train_fe_RF = X_train_fe_scaled.copy()
X_train_fe_boosting = X_train_fe_scaled.copy()

In [66]:
X_train_fe.columns

Index(['airconditioningtypeid', 'basementsqft', 'bathroomcnt', 'bedroomcnt',
       'buildingqualitytypeid', 'calculatedbathnbr',
       'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'fireplacecnt',
       'fullbathcnt', 'garagecarcnt', 'garagetotalsqft', 'hashottuborspa',
       'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet',
       'poolcnt', 'propertycountylandusecode', 'propertylandusetypeid',
       'regionidcity', 'regionidcounty', 'regionidneighborhood', 'regionidzip',
       'roomcnt', 'threequarterbathnbr', 'unitcnt', 'numberofstories',
       'taxdelinquencyflag', 'censustractandblock', 'propage',
       'livingarearatio', 'calculatedfinishedsquarefeet_log', 'roomdensity',
       'bed_bath_interaction', 'bath_per_bed', 'bedroom_intensity'],
      dtype='object')

#### Model 1: Linear Regression + Forwards and Backwards Selection

In [ ]:
# Forwards Feature Selection
sfs_linear_forward = SequentialFeatureSelector(
    linear_model,
    n_features_to_select='auto',
    tol = 100,
    direction='forward',
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=-1
)

sfs_linear_forward.fit(X_train_fe_scaled, y_train_fe)

mask = sfs_linear_forward.get_support()

selected_features = X_train_fe.columns[mask].tolist()
print("================================")
print("Forwards Selection Set:")
print(selected_features)
print("================================")

X_linear_feats_fwd = sfs_linear_forward.transform(X_train_fe_scaled)
cv_linear_fwd_scores = cross_val_score(linear_model, X_linear_feats_fwd, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("================================")
print("Linear Regression Baseline/FE, FWD FS Score")
print(f"Mean: {-cv_linear_fwd_scores.mean():.4f}")
print("================================")

Forwards Selection Set:
['bedroomcnt', 'buildingqualitytypeid', 'calculatedfinishedsquarefeet', 'fireplacecnt', 'garagetotalsqft', 'hashottuborspa', 'latitude', 'longitude', 'regionidneighborhood', 'roomcnt', 'calculatedfinishedsquarefeet_log', 'bed_bath_interaction', 'bath_per_bed', 'bedroom_intensity']
Linear Regression Baseline/FE, FWD FS Score
Mean: 168826.1312


In [ ]:
#Backwards Feature Selection
sfs_linear_backwards = SequentialFeatureSelector(
    linear_model,
    n_features_to_select='auto',
    direction='backward',
    tol=-100,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=-1
)

sfs_linear_backwards.fit(X_train_fe_scaled, y_train_fe)

mask = sfs_linear_backwards.get_support()

# Feature names directly (cleanest)
selected_features = X_train_fe.columns[mask].tolist()
print("================================")
print("Backwards Selection Set:")
print(selected_features)
print("================================")

X_linear_feats_bwd = sfs_linear_backwards.transform(X_train_fe_scaled)
cv_linear_bwd_scores = cross_val_score(linear_model, X_linear_feats_bwd, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("================================")
print("Linear Regression Baseline/FE, BWD FS Score")
print(f"Mean: {-cv_linear_bwd_scores.mean():.4f}")
print("================================")

Backwards Selection Set:
['bedroomcnt', 'buildingqualitytypeid', 'finishedsquarefeet12', 'fireplacecnt', 'garagetotalsqft', 'hashottuborspa', 'latitude', 'longitude', 'regionidneighborhood', 'roomcnt', 'calculatedfinishedsquarefeet_log', 'bed_bath_interaction', 'bedroom_intensity']
Linear Regression Baseline/FE, BWD FS Score
Mean: 168816.6773


##### Selected Model Run: Model 1

In [62]:
X_train_fe_linear = X_linear_feats_bwd
cv_linear_fs_scores = cross_val_score(linear_model, X_train_fe_linear, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Linear Regression Baseline/FE: Feature Selected Model")
print(f"Mean: {-cv_linear_fs_scores.mean():.4f}")
print(f"Std: {cv_linear_fs_scores.std():.4f}")

Linear Regression Baseline/FE: Feature Selected Model
Mean: 168816.6773
Std: 1320.3351


#### Model 2: Random Forest + Feature Importance

In [72]:
rf_model.fit(X_train_fe_scaled, y_train_fe)

rf_importance = pd.Series(rf_model.feature_importances_, index=X_train_fe.columns)
rf_importance = rf_importance.sort_values(ascending=False)

rf_importance

calculatedfinishedsquarefeet        0.125397
finishedsquarefeet12                0.116882
latitude                            0.112450
calculatedfinishedsquarefeet_log    0.111295
longitude                           0.078922
propage                             0.068954
regionidzip                         0.056580
bedroom_intensity                   0.048037
livingarearatio                     0.040769
censustractandblock                 0.039191
lotsizesquarefeet                   0.038389
buildingqualitytypeid               0.028834
regionidcity                        0.020127
regionidneighborhood                0.015821
bath_per_bed                        0.013212
garagetotalsqft                     0.011237
roomdensity                         0.009683
calculatedbathnbr                   0.009268
bathroomcnt                         0.008919
bed_bath_interaction                0.008087
propertycountylandusecode           0.006692
poolcnt                             0.005557
airconditi

In [82]:
rf_cumulative = rf_importance.cumsum()

# Keep features up to 95% cumulative importance
rf_top_features = rf_cumulative[rf_cumulative <= 0.95].index.tolist()

print(f"Features kept: {len(rf_top_features)} / {len(rf_importance)}")
print(rf_top_features)

# Filter your training data
X_train_fe_RF = X_train_fe[rf_top_features]

Features kept: 18 / 37
['calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'latitude', 'calculatedfinishedsquarefeet_log', 'longitude', 'propage', 'regionidzip', 'bedroom_intensity', 'livingarearatio', 'censustractandblock', 'lotsizesquarefeet', 'buildingqualitytypeid', 'regionidcity', 'regionidneighborhood', 'bath_per_bed', 'garagetotalsqft', 'roomdensity', 'calculatedbathnbr']


##### Selected Features Run: Model 2

In [86]:
X_train_fe_RF_scaled = scaler.fit_transform(X_train_fe_RF)
cv_rf_fs_scores = cross_val_score(rf_model, X_train_fe_RF_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Random Forest Regression Baseline + FE: Feature Selected Model")
print(f"Mean: {-cv_rf_fs_scores.mean():.4f}")
print(f"Std: {cv_rf_fs_scores.std():.4f}")

Random Forest Regression Baseline + FE: Feature Selected Model
Mean: 148796.5891
Std: 1326.1098


#### Model 3: Gradient Boosting + Feature Importance

In [87]:
boost_model.fit(X_train_fe_scaled, y_train_fe)

boost_importance = pd.Series(boost_model.feature_importances_, index=X_train_fe.columns)
boost_importance = boost_importance.sort_values(ascending=False)

boost_importance

calculatedfinishedsquarefeet        0.252088
bathroomcnt                         0.072234
latitude                            0.066838
regionidzip                         0.061769
buildingqualitytypeid               0.060738
longitude                           0.038437
propertycountylandusecode           0.037808
propage                             0.037197
hashottuborspa                      0.035295
censustractandblock                 0.030294
bath_per_bed                        0.029414
regionidcity                        0.028425
poolcnt                             0.025752
bedroom_intensity                   0.018962
propertylandusetypeid               0.018944
roomdensity                         0.017185
regionidneighborhood                0.016838
livingarearatio                     0.016137
lotsizesquarefeet                   0.014297
garagetotalsqft                     0.014049
heatingorsystemtypeid               0.013390
taxdelinquencyflag                  0.011397
roomcnt   

In [88]:
boost_cumulative = boost_importance.cumsum()

# Keep features up to 95% cumulative importance
top_features_boost = boost_cumulative[boost_cumulative <= 0.95].index.tolist()

print(f"Features kept: {len(top_features_boost)} / {len(boost_importance)}")
print(top_features_boost)

# Filter your training data
X_train_fe_boosting = X_train_fe[top_features_boost]

Features kept: 25 / 37
['calculatedfinishedsquarefeet', 'bathroomcnt', 'latitude', 'regionidzip', 'buildingqualitytypeid', 'longitude', 'propertycountylandusecode', 'propage', 'hashottuborspa', 'censustractandblock', 'bath_per_bed', 'regionidcity', 'poolcnt', 'bedroom_intensity', 'propertylandusetypeid', 'roomdensity', 'regionidneighborhood', 'livingarearatio', 'lotsizesquarefeet', 'garagetotalsqft', 'heatingorsystemtypeid', 'taxdelinquencyflag', 'roomcnt', 'unitcnt', 'fireplacecnt']


##### Selected Model Run: Model 3

In [89]:
X_train_fe_boost_scaled = scaler.fit_transform(X_train_fe_boosting)
cv_boost_fs_scores = cross_val_score(boost_model, X_train_fe_boost_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Gradient Boosting/XGBoost Regression Baseline + FE: Feature Selected Model")
print(f"Mean: {-cv_boost_fe_scores.mean():.4f}")
print(f"Std: {cv_boost_fe_scores.std():.4f}")

Gradient Boosting/XGBoost Regression Baseline + FE: Feature Selected Model
Mean: 147215.1120
Std: 1062.3372


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

#### Model 1 - Linear Regression does **NOT** have hyperparameters to tune. Moving on... 

#### Model 2 - Random Forest + Optuna

In [103]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500, step=50),
        "max_features": trial.suggest_float("max_features", 0.2, 0.8, log=False),
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 10),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1
    }

    if params["bootstrap"]:
        params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0)

    rf_hyper_model = RandomForestRegressor(**params)

    # sklearn returns NEGATIVE values for loss metrics in cross_val_score
    scores = cross_val_score(
        rf_hyper_model,
        X_train_fe_RF_scaled,
        y_train_fe,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    mae = -scores.mean()
    return mae

# ---------------------------
#  run study
# ---------------------------
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

# --- print the results ---
print("Best trial:")
print(study.best_trial.number)
print("Best MAE (CV):", study.best_value)
print("Best params:")
print(study.best_params)


Best trial: 36. Best value: 147283: 100%|██████████| 50/50 [7:37:45<00:00, 549.30s/it]  

Best trial:
36
Best MAE (CV): 147283.46409700427
Best params:
{'n_estimators': 450, 'max_features': 0.7328089307150905, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 3, 'bootstrap': True, 'max_samples': 0.8805927725981891}


In [104]:
tuned_rf_model = RandomForestRegressor(
    n_estimators=450,
    max_features=0.7328089307150905,
    max_depth=17,
    min_samples_split=6,
    min_samples_leaf=3,
    bootstrap=True,
    max_samples=0.8805927725981891,
    random_state=42,
    n_jobs=-1
)

cv_rf_tuned_scores = cross_val_score(tuned_rf_model, X_train_fe_RF_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Random Forest Regressor Hyperparamter Tuned + FE Scores")
print(f"Mean: {-cv_rf_tuned_scores.mean():.4f}")
print(f"Std: {cv_rf_tuned_scores.std():.4f}")

Random Forest Regressor Hyperparamter Tuned + FE Scores
Mean: 147283.4641
Std: 1210.5900


#### Model 3 - Gradient Boosting + Optuna

In [92]:
# Add as many cells as you need
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "device": "cpu"  
    }

    boost_hyper_model = xgb.XGBRegressor(**params)

    cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
    scores = cross_val_score(
        boost_hyper_model, X_train_fe_boost_scaled, y_train_fe,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    return scores.mean()  # Optuna maximizes by default, and neg_MAE means higher = better

# Run the study
study = optuna.create_study(direction="maximize")  # maximizing neg_MAE = minimizing MAE
study.optimize(objective, n_trials=50, show_progress_bar=True)

# Results
print(f"Best MAE: {-study.best_value:.4f}")
print(f"Best params: {study.best_params}")

Best trial: 44. Best value: -144439: 100%|██████████| 50/50 [13:29<00:00, 16.19s/it]

Best MAE: 144438.9571
Best params: {'n_estimators': 996, 'max_depth': 9, 'learning_rate': 0.011364214586861869, 'subsample': 0.9695278405010299, 'colsample_bytree': 0.5079318195172902, 'reg_alpha': 0.5968793617956106, 'reg_lambda': 0.11196214810535826, 'min_child_weight': 8}


In [ ]:
tuned_boost_model = xgb.XGBRegressor(
    n_estimators=996,
    max_depth=9,
    learning_rate=0.011364214586861869,
    subsample=0.9695278405010299,
    colsample_bytree=0.5079318195172902,
    reg_alpha=0.5968793617956106,
    reg_lambda=0.11196214810535826,
    min_child_weight=8,
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(tuned_boost_model, X_train_fe_boost_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Gradient Boosting/XGBoost Regressor Hyperparamter Tuned + FE Scores")
print(f"Mean: {-cv_scores.mean():.4f}")
print(f"Std: {cv_scores.std():.4f}")

Gradient Boosting/XGBoost Regressor Hyperparamter Tuned + FE Scores
Mean: 144438.9571
Std: 1108.5462


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [102]:
# Add as many cells as you need
final_boost_model = xgb.XGBRegressor(
    n_estimators=996,
    max_depth=9,
    learning_rate=0.011364214586861869,
    subsample=0.9695278405010299,
    colsample_bytree=0.5079318195172902,
    reg_alpha=0.5968793617956106,
    reg_lambda=0.11196214810535826,
    min_child_weight=8,
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(final_boost_model, X_train_fe_boost_scaled, y_train_fe, cv=cv, scoring='neg_mean_absolute_error')

print("Gradient Boosting/XGBoost Regressor: Final Model Scores")
print(f"Mean: {-cv_scores.mean():.4f}")
print(f"Std: {cv_scores.std():.4f}")

final_boost_model.fit(X_train_fe_boost_scaled, y_train_fe)

X_test_fe_boosting = X_test_fe[top_features_boost]

scaler.fit(X_test_fe_boosting)
X_test_fe_boosting_scaled = scaler.transform(X_test_fe_boosting)

y_pred = final_boost_model.predict(X_test_fe_boosting_scaled)

test_mae = mean_absolute_error(y_test_fe, y_pred)
test_median_ae = median_absolute_error(y_test_fe, y_pred)

print(f"Test MAE Score: {test_mae:.4f}")
print(f"Test Median AE Score: {test_median_ae:.4f}")

Gradient Boosting/XGBoost Regressor: Final Model Scores
Mean: 144438.9571
Std: 1108.5462
Test MAE Score: 154576.3783
Test Median AE Score: 114676.6250


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here